In [1]:
# Cell 1
# WHAT: mount Google Drive, find the research-engine folder and move into it.
# WHY: all code, data caches and results live in the Drive copy of the repo.
#      The search avoids guessing the exact path.
from google.colab import drive
import glob, os
drive.mount('/content/drive')
hits = [p for p in glob.glob('/content/drive/MyDrive/**/agent/config.py', recursive=True) if 'research' in p.lower()]
print(hits)
PROJECT = os.path.dirname(os.path.dirname(hits[0]))
os.chdir(PROJECT)
print("now in:", os.getcwd())
!git log --oneline -1                        # should show 3b8e6dd

Mounted at /content/drive
['/content/drive/MyDrive/research-engine/agent/config.py']
now in: /content/drive/MyDrive/research-engine
e4e452e (HEAD -> main) Day 10: FAISS index for corpus-wide dense search; benchmark vs numpy (flat exact, HNSW speed/recall grid)


In [2]:
# Cell 3
# WHAT: install requirements (now including google-genai, the Gemini SDK).
# WHY: needed after every runtime restart; google-genai is new today.
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.0 MB/s eta 0:00:00


In [3]:
# Cell 4
# WHAT: copy both API keys from Colab Secrets into environment variables.
# WHY: `!python` commands run in a separate process and only see environment variables, not Colab secrets.
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY").strip()
print("keys set")

keys set


In [ ]:
# Cell 5
# WHAT: check that the key works and that the model ID exists, with one tiny Gemini call.
# WHY: catches a wrong key or a renamed model before a 50-question run.
from google import genai
from agent.llm import make_llm
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print([m.name for m in client.models.list() if "flash" in m.name])
llm = make_llm("gemini-3.8-flash")
text, usage = llm.chat([{"role": "user", "content": 'Reply with JSON {"ok": true}'}], max_tokens=256, json_mode=True)
print(text, usage)

['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-3.8-flash-tts', 'models/gemini-3.8-flash-lite-tts', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-preview-12-2025', 'models/gemini-3.1-flash-live-preview']
{"ok": true} {'prompt_tokens': 9, 'completion_tokens': 5, 'secs': 37.04, 

In [ ]:
# Cell 6
# WHAT: apply the reference-quality filter to the saved n=50 results (no API calls).
# WHY: checks the filter in Colab: it should exclude Q27, Q39, Q45 and print full vs filtered metrics.
!python -m eval.reference_filter results/agent_eval_dev_n50_v2_judge-gpt-oss-20b.jsonl

Rule: exclude a question if ALL its references either start with 'Answer with content missing' or contain a BIBREF/FIGREF/TABREF/SECREF/EQREF marker and no content word besides markers and filler words
50 questions -> 47 kept, 3 excluded
  Q27: what dataset is used in this paper?  (only a citation/figure/table marker)
  Q39: What are the state of the art models?  (only a citation/figure/table marker)
  Q45: What are the baselines?  (content missing)

| metric | full (n=50) | filtered (n=47) |
|---|---|---|
| strict_citation_precision | 0.6278 | 0.6193 |
| judged_citation_precision | 0.8787 | 0.8759 |
| faithfulness | 0.8778 | 0.875 |
| answer_correctness | 0.68 | 0.7234 |
| answer_correctness_answered_only | 0.7556 | 0.7727 |
| declined_rate | 0.1 | 0.0638 |
| judge_errors | 0 | 0 |
| avg_judge_prompt_tokens | 1109 | 1113 |
| avg_judge_completion_tokens | 170 | 170 |


In [ ]:
# Cell 7
# WHAT: run Gemini (gemini-3.8-flash) as a second judge on the saved n=20 v2 agent answers.
# WHY: a judge from a different model family than the gpt-oss agent. If it agrees with the
#      gpt-oss-20b judge, the judged metrics (citation precision 0.94 on n=20) are more trustworthy.
#      It reads the saved .jsonl only: no agent calls and no Groq tokens.
#      Output: results/agent_eval_dev_n20_v2_judge-gemini-38-flash.json(l). If it stops on the daily
#      quota, run the same command with --resume later.
!python -m eval.judge_agent_eval results/agent_eval_dev_n20_v2.jsonl --model gemini-3.8-flash --resume

20 answers in agent_eval_dev_n20_v2.jsonl | judge=gemini-3.8-flash
  (Gemini 503: waiting 5s, retry 1/8)
[  1/20] strict=1.00 judged=1.00 faithful=yes correct=yes  What measures are used for evaluation?
  (Gemini 503: waiting 5s, retry 1/8)
  (Gemini 503: waiting 10s, retry 2/8)
[  2/20] strict=1.00 judged=1.00 faithful=yes correct=yes  In what way is the offensive dataset not biased by topi
[  3/20] strict=0.00 judged=1.00 faithful=yes correct=partly  by how much did nus outperform abus?
[  4/20] strict=0.50 judged=1.00 faithful=yes correct=yes  What background knowledge do they leverage?
  (Gemini 503: waiting 5s, retry 1/8)
  (Gemini 503: waiting 10s, retry 2/8)
  (Gemini 503: waiting 20s, retry 3/8)
  (Gemini 503: waiting 40s, retry 4/8)

STOPPED at 5/20: daily quota reached for gemini-3.8-flash. 4 judged rows saved; run again later with --resume.

judge=gemini-3.8-flash  n=4  (PARTIAL: 4 of 20; finish with --resume)
| metric | full (n=4) | reference-filtered (n=4) |
|---|---|---|


In [ ]:
# Cell 8
# WHAT: compare the two judges on the same 20 answers (agreement, Cohen's kappa, every disagreement).
# WHY: shows how much the judged metrics depend on which LLM does the judging.
#      Also writes results/judge_agreement_agent_eval_dev_n20_v2_gpt-oss-20b_vs_gemini-38-flash.md
!python -m eval.compare_judges results/agent_eval_dev_n20_v2_judge-gpt-oss-20b.jsonl results/agent_eval_dev_n20_v2_judge-gemini-38-flash.jsonl

In [ ]:
# Cell 10
# WHAT: print the exact error that stopped the Gemini judge.
# WHY: tells us whether it was the per-DAY quota (wait until it resets) or a per-MINUTE limit
#      that the code mistook for a daily one (then I fix the detection).
import json
s = json.load(open("results/agent_eval_dev_n20_v2_judge-gemini-38-flash.json"))
print(s["stopped_reason"])

429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 4h30m13.878935355s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.8-flash'}, 'quotaV

In [ ]:
!python -m eval.judge_agent_eval results/agent_eval_dev_n50_v2.jsonl --model gemini-3.8-flash --resume

50 answers in agent_eval_dev_n50_v2.jsonl | judge=gemini-3.8-flash
  (Gemini 503: waiting 5s, retry 1/6)
  (Gemini 503: waiting 10s, retry 2/6)

STOPPED at 1/50: the model is overloaded (5xx after retries) for gemini-3.8-flash. 0 judged rows saved; run again later with --resume.
Nothing judged.


In [ ]:
# Cell 14
# WHAT: Gemini second judge on the saved dev n=50 agent answers, using gemini-3.6-flash.
# WHY: 3.8 Flash is congested (24 s for a one-word reply, frequent 503s that use up the 20/day quota);
#      3.6 Flash answers in ~2 s. One judge model for all 50 questions keeps the comparison clean.
#      About 18-19 questions per day: it stops cleanly at the daily limit; rerun with --resume tomorrow.
#      Output: results/agent_eval_dev_n50_v2_judge-gemini-36-flash.json(l)
!python -m eval.judge_agent_eval results/agent_eval_dev_n50_v2.jsonl --model gemini-3.6-flash --resume

50 answers in agent_eval_dev_n50_v2.jsonl | judge=gemini-3.6-flash
[  1/50] strict=1.00 judged=1.00 faithful=yes correct=yes  What measures are used for evaluation?
  (Gemini 503: waiting 5s, retry 1/6)
  (Gemini 503: waiting 10s, retry 2/6)

STOPPED at 2/50: the model is overloaded (5xx after retries) for gemini-3.6-flash. 1 judged rows saved; run again later with --resume.

judge=gemini-3.6-flash  n=1  (PARTIAL: 1 of 50; finish with --resume)
| metric | full (n=1) | reference-filtered (n=1) |
|---|---|---|
| strict_citation_precision | 1.0 | 1.0 |
| judged_citation_precision | 1.0 | 1.0 |
| faithfulness | 1.0 | 1.0 |
| answer_correctness | 1.0 | 1.0 |
| answer_correctness_answered_only | 1.0 | 1.0 |
| declined_rate | 0.0 | 0.0 |
| judge_errors | 0 | 0 |
| avg_judge_prompt_tokens | 570 | 570 |
| avg_judge_completion_tokens | 49 | 49 |

Saved results/agent_eval_dev_n50_v2_judge-gemini-36-flash.json and results/agent_eval_dev_n50_v2_judge-gemini-36-flash.jsonl


In [5]:
# Cell 15
# WHAT: run the Gemini judge in a loop: after an "overloaded" stop, wait 1 hour and resume;
#       finish when the daily quota is used up or all 50 questions are judged.
# WHY: Gemini's free tier often answers 503 "high demand"; the judge stops early to save quota.
#      This loop restarts it automatically instead of you re-running the cell every hour.
#      Keep this Colab tab open (Colab may disconnect an idle runtime after ~90 min without a visible tab).
import json, subprocess, time, datetime
CMD = ["python", "-m", "eval.judge_agent_eval", "results/agent_eval_dev_n50_v2.jsonl",
       "--model", "gemini-3.6-flash", "--resume"]
SUMMARY = "results/agent_eval_dev_n50_v2_judge-gemini-36-flash.json"
for attempt in range(1, 11):                      # at most 10 tries (~10 hours)
    print(f"\n=== run {attempt} at {datetime.datetime.now():%H:%M} ===")
    out = subprocess.run(CMD, capture_output=True, text=True)
    print("\n".join(l for l in out.stdout.splitlines() if l.startswith("[") or "STOPPED" in l))
    if out.returncode != 0:                       # a real error, not a quota stop: show it and stop
        print(out.stderr[-2000:]); break
    s = json.load(open(SUMMARY))
    print(f"judged so far: {s['n']}/{s['n_planned']}")
    if s["complete"]:
        print("ALL DONE"); break
    reason = (s.get("stopped_reason") or "").lower()
    if "perday" in reason or "per day" in reason:
        print("Daily quota used up: run this cell again tomorrow."); break
    print("Model overloaded: waiting 60 minutes...")
    time.sleep(3600)


=== run 1 at 07:01 ===
[ 42/50] strict=1.00 judged=1.00 faithful=yes correct=yes  How they extract "structured answer-relevant relation"?
[ 43/50] strict=1.00 judged=1.00 faithful=yes correct=yes  How are the accuracy merits of the approach demonstrate
[ 44/50] strict=0.67 judged=1.00 faithful=yes correct=yes  What metrics are used for evaluation?
[ 45/50] strict=- judged=- faithful=- correct=no  What are the baselines?
[ 46/50] strict=1.00 judged=1.00 faithful=yes correct=yes  What sentiment analysis dataset is used?
[ 47/50] strict=0.00 judged=1.00 faithful=yes correct=yes  did they compare with other extractive summarization me
[ 48/50] strict=1.00 judged=0.50 faithful=partly correct=no  How do their models decide how much improtance to give 
[ 49/50] strict=0.00 judged=1.00 faithful=yes correct=yes  What kind of issues (that are not on the forefront of c
[ 50/50] strict=0.00 judged=1.00 faithful=yes correct=partly  What dataset is used?
judged so far: 50/50
ALL DONE


In [6]:
# Cell 16
# WHAT: compare the two judges (gpt-oss-20b vs gemini-3.6-flash) on the same 50 dev answers.
# WHY: measures how much the judged metrics depend on which LLM judges: agreement, Cohen's kappa,
#      both judges' metrics side by side (full and reference-filtered), and every disagreement.
#      Writes results/judge_agreement_agent_eval_dev_n50_v2_gpt-oss-20b_vs_gemini-36-flash.json/.md
!python -m eval.compare_judges results/agent_eval_dev_n50_v2_judge-gpt-oss-20b.jsonl results/agent_eval_dev_n50_v2_judge-gemini-36-flash.jsonl

# Judge agreement: openai/gpt-oss-20b (A) vs gemini-3.6-flash (B)

Agent results: `agent_eval_dev_n50_v2`; 50 questions, 45 answered.

| verdict | items | exact agreement | Cohen's kappa | lenient agreement (yes/partly vs no) |
|---|---|---|---|---|
| correct | 45 | 0.822 | 0.498 | 0.889 |
| faithful | 45 | 0.822 | 0.151 | 0.933 |
| citations | 72 | 0.819 | 0.037 | 0.889 |

| metric | A full | B full | A filtered | B filtered |
|---|---|---|---|---|
| strict_citation_precision | 0.6278 | 0.6278 | 0.6193 | 0.6193 |
| judged_citation_precision | 0.8787 | 0.9778 | 0.8759 | 0.9773 |
| faithfulness | 0.8778 | 0.9778 | 0.875 | 0.9773 |
| answer_correctness | 0.68 | 0.78 | 0.7234 | 0.8085 |
| answer_correctness_answered_only | 0.7556 | 0.8667 | 0.7727 | 0.8636 |
| judge_errors | 0 | 0 | 0 | 0 |
| n | 50 | 50 | 47 | 47 |

## Disagreements (15)

- **Q3** by how much did nus outperform abus?  
  correct A/B = no/yes, faithful = yes/yes  
  A: Answer differs from reference by 0.8 percentage point